# Synthetic Patient Data Generator (Simplified)

Generates 20 fictional patient profiles for testing the breast-cancer clinical
trial eligibility matcher (Agent 2). This is the **simplified** field set:
diagnosis, stage, ER/PR/HER2 biomarkers, ECOG performance status, prior
treatments, and 2 key lab values.

A handful of patients have specific fields **deliberately left missing**
(set to `null`), so Agent 2's "Unknown" handling actually gets exercised
during testing, not just the happy-path Met/Not Met cases.

In [10]:
import json
import random
import pandas as pd

PATIENT_SEED = 42
random.seed(PATIENT_SEED)

N_PATIENTS = 20


In [11]:
DIAGNOSES = [
    "Invasive ductal carcinoma",
    "Invasive lobular carcinoma",
    "Ductal carcinoma in situ (DCIS)",
    "Triple-negative breast carcinoma",
    "Inflammatory breast carcinoma",
]

STAGES = ["0", "I", "II", "III", "IV"]
STAGE_WEIGHTS = [0.05, 0.25, 0.30, 0.20, 0.20]

ECOG_WEIGHTS = {0: 0.35, 1: 0.35, 2: 0.15, 3: 0.10, 4: 0.05}

PRIOR_TREATMENT_OPTIONS = [
    "surgery", "chemotherapy", "radiation therapy",
    "hormone therapy", "targeted therapy (anti-HER2)", "immunotherapy",
]


def weighted_choice(options, weights):
    return random.choices(options, weights=weights, k=1)[0]


In [12]:
def random_key_labs(impaired: bool):
    """Just the 2 most commonly-checked lab values across breast-cancer trials."""
    if impaired:
        return {
            "absolute_neutrophil_count_per_uL": round(random.uniform(800, 1400), 0),  # normal >=1500
            "platelet_count_per_uL": round(random.uniform(60000, 95000), 0),          # normal >=100000
        }
    return {
        "absolute_neutrophil_count_per_uL": round(random.uniform(1600, 6500), 0),
        "platelet_count_per_uL": round(random.uniform(150000, 380000), 0),
    }


In [13]:
def make_patient(idx: int):
    patient_id = f"SYN-{idx:03d}"

    age = random.randint(28, 82)
    sex = "Female" if random.random() > 0.03 else "Male"
    if sex == "Female":
        menopausal_status = "Postmenopausal" if age >= 52 else random.choice(
            ["Premenopausal", "Perimenopausal"]
        )
    else:
        menopausal_status = "N/A"

    diagnosis = random.choice(DIAGNOSES)
    stage = weighted_choice(STAGES, STAGE_WEIGHTS)

    if diagnosis == "Triple-negative breast carcinoma":
        er_status = pr_status = "Negative"
        her2_status = "Negative"
    else:
        er_status = random.choice(["Positive", "Negative"])
        pr_status = random.choice(["Positive", "Negative"])
        her2_status = random.choices(
            ["Positive", "Negative", "Equivocal"], weights=[0.2, 0.7, 0.1]
        )[0]

    ecog = random.choices(list(ECOG_WEIGHTS.keys()), weights=list(ECOG_WEIGHTS.values()))[0]

    n_prior = random.randint(0, 3)
    prior_treatments = random.sample(PRIOR_TREATMENT_OPTIONS, k=n_prior) if n_prior else ["none"]

    impaired_organ_function = random.random() < 0.25
    key_labs = random_key_labs(impaired_organ_function)

    return {
        "patient_id": patient_id,
        "age": age,
        "sex": sex,
        "menopausal_status": menopausal_status,
        "diagnosis": diagnosis,
        "cancer_stage": stage,
        "biomarkers": {
            "ER_status": er_status,
            "PR_status": pr_status,
            "HER2_status": her2_status,
        },
        "ecog_performance_status": ecog,
        "prior_treatments": prior_treatments,
        "key_lab_values": key_labs,
    }


In [14]:
patients = [make_patient(i) for i in range(1, N_PATIENTS + 1)]
print(f"Generated {len(patients)} synthetic patients (simplified)")

Generated 20 synthetic patients (simplified)


## Deliberately introduce missing information

Real clinical records are sometimes incomplete, and Agent 2 (the matcher) is
built to handle that gracefully by returning `"Unknown"` for any rule whose
required field is missing. To make sure that path is actually tested (not
just theoretically supported), a few patients below have specific fields
deliberately set to `null`.

The plan is fixed (not random) so it stays reproducible, and it's applied
*after* generation so it doesn't disturb the random sequence used for
everyone else's data.

In [15]:
# Fixed (not random) so it's reproducible and easy to reason about. Covers
# a mix of field types so each of Agent 2's checkers gets a real "Unknown"
# test case: a numeric field, a categorical field, a list-type field used
# by both categorical_contains and text_interpretation rules, and an
# entire nested lab-values block missing.
MISSING_INFO_PLAN = {
    "SYN-005": ["ecog_performance_status"],   # numeric check -> Unknown
    "SYN-011": ["biomarkers.HER2_status"],    # categorical check -> Unknown
    "SYN-016": ["prior_treatments"],          # categorical_contains / text_interpretation -> Unknown
    "SYN-020": ["key_lab_values"],            # whole lab panel missing -> Unknown
}


def apply_missing_info(patient_list, plan):
    by_id = {p["patient_id"]: p for p in patient_list}
    for patient_id, dotted_fields in plan.items():
        patient = by_id.get(patient_id)
        if patient is None:
            continue
        for dotted_field in dotted_fields:
            parts = dotted_field.split(".")
            target = patient
            for part in parts[:-1]:
                target = target[part]
            target[parts[-1]] = None
    return patient_list


patients = apply_missing_info(patients, MISSING_INFO_PLAN)
print(f"Deliberately missing fields applied to: {', '.join(MISSING_INFO_PLAN.keys())}")

Deliberately missing fields applied to: SYN-005, SYN-011, SYN-016, SYN-020


In [16]:
out_path = "synthetic_patients_simple.json"
with open(out_path, "w") as f:
    json.dump(patients, f, indent=2)

print(f"Saved {len(patients)} patients -> {out_path}")

Saved 20 patients -> synthetic_patients_simple.json


## Preview the generated data

Run this to double check the output looks right before committing it to the repo.


In [17]:
with open("synthetic_patients_simple.json") as f:
    data = json.load(f)

print(len(data), "patients")
df = pd.json_normalize(data)
df.head()

20 patients


,patient_id,age,sex,menopausal_status,diagnosis,cancer_stage,ecog_performance_status,prior_treatments,biomarkers.ER_status,biomarkers.PR_status,biomarkers.HER2_status,key_lab_values.absolute_neutrophil_count_per_uL,key_lab_values.platelet_count_per_uL,key_lab_values
0,SYN-001,68,Female,Postmenopausal,Ductal carcinoma in situ (DCIS),I,3.0,[none],Positive,Positive,Negative,1756.0,171550.0,NaN
1,SYN-002,42,Female,Premenopausal,Inflammatory breast carcinoma,I,0.0,[none],Negative,Positive,Negative,2382.0,247201.0,NaN
2,SYN-003,45,Female,Perimenopausal,Invasive ductal carcinoma,I,1.0,[none],Positive,Negative,Negative,4228.0,373817.0,NaN
3,SYN-004,52,Female,Postmenopausal,Ductal carcinoma in situ (DCIS),IV,0.0,[radiation therapy],Negative,Positive,Negative,5791.0,349291.0,NaN
4,SYN-005,52,Female,Postmenopausal,Ductal carcinoma in situ (DCIS),I,NaN,[none],Negative,Positive,Negative,2439.0,317699.0,NaN
